# ASSIGNMENT 1 - Question 4 - Handwritten Postal-Code Reader
### Alumno: Martín Andrés Gutiérrez López
Topics Max & Average Pooling, Lenet-5

In [2]:
# ==============================================================================
# Question 4: Handwritten Postal-Code Reader (Case: PostSort)
# Verification of Pooling Operations and LeNet-5 Architecture Parameters
# ==============================================================================

import numpy as np
import torch
import torch.nn as nn

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Part (a): Pooling operations on the 4x4 feature map F
# ------------------------------------------------------------------------------
# Feature map F
F_np = np.array(
    [[1, 3, 2, 4], [5, 6, 1, 2], [7, 2, 9, 0], [1, 8, 3, 4]], dtype=float
)

# Convert to PyTorch tensor with shape (Batch=1, Channels=1, Height=4, Width=4)
F_tensor = torch.tensor(F_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Define 2x2 pooling layers with stride 2
max_pool_2d = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool_2d = nn.AvgPool2d(kernel_size=2, stride=2)

# Compute pooling outputs
max_pooled = max_pool_2d(F_tensor).squeeze().numpy()
avg_pooled = avg_pool_2d(F_tensor).squeeze().numpy()

print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("PART (a): POOLING COMPARISON (2x2, stride=2)")
print("=" * 70,f"{RESET}")
print("Input Feature Map F (4x4):")
print(F_np.astype(int))
print("\n2x2 Max Pooling Output:")
print(max_pooled.astype(int))
print("\n2x2 Average Pooling Output:")
print(avg_pooled)
print(
    "\nInterpretation: Max pooling better preserves the strongest stroke"
    " activations\nbecause it selects the peak feature response, whereas"
    " average pooling\ndilutes the stroke signal with background zeros/low"
    " values."
)


# ------------------------------------------------------------------------------
# Part (b): LeNet-5 Architecture, Output Shapes, and Parameter Calculation
# ------------------------------------------------------------------------------
class LeNet5PostSort(nn.Module):

  def __init__(self):
    super().__init__()
    # C1: 6 filters, 5x5, stride 1 (Input 32x32x1)
    self.c1 = nn.Conv2d(
        in_channels=1, out_channels=6, kernel_size=5, stride=1, bias=True
    )
    # S2: 2x2 avg pool, stride 2 (parameter-free)
    self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
    # C3: 16 filters, 5x5, stride 1 (fully connected to all 6 S2 channels)
    self.c3 = nn.Conv2d(
        in_channels=6, out_channels=16, kernel_size=5, stride=1, bias=True
    )
    # S4: 2x2 avg pool, stride 2 (parameter-free)
    self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
    # C5: 120 filters, 5x5, stride 1 (convolutional flattening layer)
    self.c5 = nn.Conv2d(
        in_channels=16, out_channels=120, kernel_size=5, stride=1, bias=True
    )
    # F6: Fully connected layer, 84 units
    self.f6 = nn.Linear(in_features=120, out_features=84, bias=True)
    # Output: Fully connected layer, 10 units
    self.output = nn.Linear(in_features=84, out_features=10, bias=True)

  def forward(self, x):
    out_c1 = self.c1(x)
    out_s2 = self.s2(out_c1)
    out_c3 = self.c3(out_s2)
    out_s4 = self.s4(out_c3)
    out_c5 = self.c5(out_s4)
    flat_c5 = out_c5.view(out_c5.size(0), -1)
    out_f6 = self.f6(flat_c5)
    out_final = self.output(out_f6)
    return {
        "C1": out_c1,
        "S2": out_s2,
        "C3": out_c3,
        "S4": out_s4,
        "C5": out_c5,
        "F6": out_f6,
        "Output": out_final,
    }


model = LeNet5PostSort()

# Dummy input matching LeNet-5 specification: (Batch=1, Channels=1, Height=32, Width=32)
dummy_input = torch.randn(1, 1, 32, 32)
layer_outputs = model(dummy_input)

# Layer-by-layer specifications for table printing
table_metadata = [
    {
        "layer": "Input",
        "operation": "—",
        "shape": "32x32x1",
        "params": 0,
        "is_none": True,
    },
    {
        "layer": "C1",
        "operation": "6 filters, 5x5, stride 1",
        "key": "C1",
        "module": model.c1,
    },
    {
        "layer": "S2",
        "operation": "2x2 avg pool, stride 2",
        "key": "S2",
        "module": None,
    },
    {
        "layer": "C3",
        "operation": "16 filters, 5x5, stride 1",
        "key": "C3",
        "module": model.c3,
    },
    {
        "layer": "S4",
        "operation": "2x2 avg pool, stride 2",
        "key": "S4",
        "module": None,
    },
    {
        "layer": "C5",
        "operation": "120 filters, 5x5",
        "key": "C5",
        "module": model.c5,
    },
    {
        "layer": "F6",
        "operation": "Fully connected, 84 units",
        "key": "F6",
        "module": model.f6,
    },
    {
        "layer": "Output",
        "operation": "Fully connected, 10 units",
        "key": "Output",
        "module": model.output,
    },
]

print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("PART (b): COMPLETED LeNet-5 TABLE & PARAMETER COUNT")
print("=" * 70,f"{RESET}")
print(f"{'Layer':<8} | {'Operation':<30} | {'Output shape':<15} | {'Parameters':<10}")
print("-" * 70)

total_params = 0

for item in table_metadata:
  if item.get("is_none", False):
    print(
        f"{item['layer']:<8} | {item['operation']:<30} | {item['shape']:<15} |"
        " —"
    )
    continue

  # Extract output tensor shape
  out_tensor = layer_outputs[item["key"]]
  if len(out_tensor.shape) == 4:
    # (Batch, Channels, H, W) -> HxWxC
    shape_str = (
        f"{out_tensor.shape[2]}x{out_tensor.shape[3]}x{out_tensor.shape[1]}"
    )
  else:
    # 1D vector (F6 or Output)
    shape_str = f"{out_tensor.shape[1]}"

  # Count parameters
  if item["module"] is not None:
    p_count = sum(p.numel() for p in item["module"].parameters())
  else:
    p_count = 0

  total_params += p_count
  print(
      f"{item['layer']:<8} | {item['operation']:<30} | {shape_str:<15} |"
      f" {p_count:<10}"
  )

print("-" * 70)
print(f"Total Learnable Parameters: {total_params:,}")



PART (a): POOLING COMPARISON (2x2, stride=2)
Input Feature Map F (4x4):
[[1 3 2 4]
 [5 6 1 2]
 [7 2 9 0]
 [1 8 3 4]]

2x2 Max Pooling Output:
[[6 4]
 [8 9]]

2x2 Average Pooling Output:
[[3.75 2.25]
 [4.5  4.  ]]

Interpretation: Max pooling better preserves the strongest stroke activations
because it selects the peak feature response, whereas average pooling
dilutes the stroke signal with background zeros/low values.


PART (b): COMPLETED LeNet-5 TABLE & PARAMETER COUNT
Layer    | Operation                      | Output shape    | Parameters
----------------------------------------------------------------------
Input    | —                              | 32x32x1         | —
C1       | 6 filters, 5x5, stride 1       | 28x28x6         | 156       
S2       | 2x2 avg pool, stride 2         | 14x14x6         | 0         
C3       | 16 filters, 5x5, stride 1      | 10x10x16        | 2416      
S4       | 2x2 avg pool, stride 2         | 5x5x16          | 0         
C5       | 120 filters